# Module 0: Feature Selection Methods Comparison

## Learning Objectives

By completing this notebook, you will:
1. Understand the three main categories of feature selection methods
2. Implement filter methods (correlation, mutual information, chi-square)
3. Implement wrapper methods (recursive feature elimination)
4. Implement embedded methods (Lasso, tree-based importance)
5. Compare all methods on a real dataset
6. Identify when to use genetic algorithms for feature selection

## Prerequisites

- Python basics and NumPy
- Understanding of classification and regression
- Familiarity with scikit-learn

## Estimated Time: 60 minutes

---

## 1. Setup and Data Preparation

We'll use a breast cancer dataset with 30 features. Some features are highly correlated, making this ideal for comparing feature selection methods.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

### 1.1 Load and Explore Dataset

The breast cancer dataset contains features computed from digitized images of breast mass. Our goal is to predict whether a tumor is malignant or benign.

In [ ]:
# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print(f"Dataset shape: {X.shape}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"\nClass distribution:")
print(f"  Malignant (0): {np.sum(y == 0)}")
print(f"  Benign (1): {np.sum(y == 1)}")
print(f"\nFeature names:")
for i, name in enumerate(data.feature_names[:5]):
    print(f"  {i}: {name}")
print(f"  ... ({len(data.feature_names) - 5} more features)")

### 1.2 Split and Standardize Data

Feature selection should be performed on training data only to avoid data leakage.

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features (important for many methods)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"\nFeature statistics after scaling (train):")
print(f"  Mean: {X_train_scaled.mean().mean():.4f}")
print(f"  Std: {X_train_scaled.std().mean():.4f}")

## 2. Filter Methods

### Key Concept: Filter methods rank features based on statistical measures, independent of the machine learning model.

**Advantages:**
- Fast computation
- Model-agnostic
- Good for high-dimensional data

**Disadvantages:**
- Ignores feature interactions
- May select redundant features

### 2.1 Correlation-Based Selection

Select features with high absolute correlation with the target variable.

In [ ]:
# Purpose: Calculate point-biserial correlation for binary target
# Key Concept: Higher absolute correlation indicates stronger linear relationship

from scipy.stats import pearsonr

def correlation_filter(X, y, k=10):
    """
    Select top k features based on correlation with target.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    k : int
        Number of features to select
    
    Returns
    -------
    selected_features : list
        Names of selected features
    scores : Series
        Correlation scores for all features
    """
    # Calculate correlation for each feature
    correlations = {}
    for col in X.columns:
        corr, _ = pearsonr(X[col], y)
        correlations[col] = abs(corr)  # Use absolute value
    
    # Convert to Series and sort
    scores = pd.Series(correlations).sort_values(ascending=False)
    
    # Select top k features
    selected_features = scores.head(k).index.tolist()
    
    return selected_features, scores

# Apply correlation filter
k_features = 10
corr_features, corr_scores = correlation_filter(X_train_scaled, y_train, k=k_features)

print(f"Top {k_features} features by correlation:")
for i, feat in enumerate(corr_features, 1):
    print(f"  {i}. {feat}: {corr_scores[feat]:.4f}")

### 2.2 Mutual Information

Mutual information measures the reduction in uncertainty about one variable given knowledge of another. It captures non-linear relationships better than correlation.

In [ ]:
# Purpose: Use mutual information to capture non-linear relationships
# Key Concept: MI = 0 means independent, higher values mean stronger dependency

from sklearn.feature_selection import mutual_info_classif

def mutual_info_filter(X, y, k=10):
    """
    Select top k features based on mutual information.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    k : int
        Number of features to select
    
    Returns
    -------
    selected_features : list
        Names of selected features
    scores : Series
        MI scores for all features
    """
    # Calculate mutual information
    mi_scores = mutual_info_classif(X, y, random_state=42)
    
    # Create Series and sort
    scores = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
    
    # Select top k features
    selected_features = scores.head(k).index.tolist()
    
    return selected_features, scores

# Apply MI filter
mi_features, mi_scores = mutual_info_filter(X_train_scaled, y_train, k=k_features)

print(f"Top {k_features} features by mutual information:")
for i, feat in enumerate(mi_features, 1):
    print(f"  {i}. {feat}: {mi_scores[feat]:.4f}")

### 2.3 Variance Threshold

Remove features with low variance (near-constant features provide little information).

In [ ]:
# Purpose: Identify and remove low-variance features
# Key Concept: Low variance means feature values don't vary much

from sklearn.feature_selection import VarianceThreshold

def variance_filter(X, threshold=0.1):
    """
    Remove features with variance below threshold.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix (should be scaled)
    threshold : float
        Minimum variance threshold
    
    Returns
    -------
    selected_features : list
        Names of features above threshold
    variances : Series
        Variance of all features
    """
    # Calculate variances
    variances = X.var().sort_values(ascending=False)
    
    # Select features above threshold
    selected_features = variances[variances > threshold].index.tolist()
    
    return selected_features, variances

# Apply variance filter
var_features, variances = variance_filter(X_train_scaled, threshold=0.5)

print(f"Features with variance > 0.5: {len(var_features)}/{len(X.columns)}")
print(f"\nTop 10 features by variance:")
for i, feat in enumerate(variances.head(10).index, 1):
    print(f"  {i}. {feat}: {variances[feat]:.4f}")

## 3. Wrapper Methods

### Key Concept: Wrapper methods use a machine learning model to evaluate feature subsets.

**Advantages:**
- Consider feature interactions
- Optimized for specific model
- Often achieve better performance

**Disadvantages:**
- Computationally expensive
- Risk of overfitting
- Model-dependent

### 3.1 Recursive Feature Elimination (RFE)

RFE recursively removes the least important features based on model coefficients or feature importances.

In [ ]:
# Purpose: Use RFE to iteratively remove least important features
# Key Concept: Train model, remove worst feature, repeat until k features remain

from sklearn.feature_selection import RFE

def rfe_wrapper(X, y, k=10):
    """
    Select top k features using Recursive Feature Elimination.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    k : int
        Number of features to select
    
    Returns
    -------
    selected_features : list
        Names of selected features
    ranking : Series
        Feature ranking (1 = selected)
    """
    # Use logistic regression as the estimator
    estimator = LogisticRegression(max_iter=1000, random_state=42)
    
    # Create RFE selector
    selector = RFE(estimator, n_features_to_select=k, step=1)
    selector.fit(X, y)
    
    # Get selected features
    selected_mask = selector.support_
    selected_features = X.columns[selected_mask].tolist()
    
    # Get ranking (1 = best)
    ranking = pd.Series(selector.ranking_, index=X.columns).sort_values()
    
    return selected_features, ranking

# Apply RFE
rfe_features, rfe_ranking = rfe_wrapper(X_train_scaled, y_train, k=k_features)

print(f"Top {k_features} features by RFE:")
for i, feat in enumerate(rfe_features, 1):
    print(f"  {i}. {feat}")
print(f"\nNote: All selected features have rank = 1")

### 3.2 Forward Selection (Simplified)

Start with no features and iteratively add the feature that most improves model performance.

In [ ]:
# Purpose: Implement greedy forward feature selection
# Key Concept: Start empty, add best feature one at a time

def forward_selection(X, y, k=10, cv=5):
    """
    Select k features using greedy forward selection.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    k : int
        Number of features to select
    cv : int
        Number of cross-validation folds
    
    Returns
    -------
    selected_features : list
        Names of selected features in order added
    scores : list
        CV score after adding each feature
    """
    # Initialize
    selected_features = []
    remaining_features = X.columns.tolist()
    scores = []
    
    # Greedy selection
    for i in range(k):
        best_score = -np.inf
        best_feature = None
        
        # Try adding each remaining feature
        for feature in remaining_features:
            # Create feature subset
            features_to_try = selected_features + [feature]
            X_subset = X[features_to_try]
            
            # Evaluate with cross-validation
            model = LogisticRegression(max_iter=1000, random_state=42)
            score = cross_val_score(model, X_subset, y, cv=cv, scoring='accuracy').mean()
            
            # Track best feature
            if score > best_score:
                best_score = score
                best_feature = feature
        
        # Add best feature
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        scores.append(best_score)
        
        print(f"Step {i+1}: Added '{best_feature}' (CV accuracy: {best_score:.4f})")
    
    return selected_features, scores

# Apply forward selection (warning: slow!)
print(f"Running forward selection for {k_features} features...\n")
forward_features, forward_scores = forward_selection(
    X_train_scaled, y_train, k=k_features, cv=3  # Use cv=3 for speed
)

## 4. Embedded Methods

### Key Concept: Embedded methods perform feature selection as part of the model training process.

**Advantages:**
- Less computationally expensive than wrappers
- Consider feature interactions
- Regularization prevents overfitting

**Disadvantages:**
- Model-specific
- May require hyperparameter tuning

### 4.1 Lasso (L1 Regularization)

Lasso regression drives some coefficients to exactly zero, effectively performing feature selection.

In [ ]:
# Purpose: Use L1 regularization to shrink coefficients to zero
# Key Concept: Lasso penalty forces sparse solutions

from sklearn.linear_model import LogisticRegression

def lasso_embedded(X, y, k=10, C=0.1):
    """
    Select features using Lasso (L1) regularization.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    k : int
        Number of features to select
    C : float
        Inverse regularization strength (smaller = more regularization)
    
    Returns
    -------
    selected_features : list
        Names of selected features
    coefficients : Series
        Absolute coefficient values
    """
    # Train Lasso model
    model = LogisticRegression(
        penalty='l1',
        C=C,
        solver='liblinear',
        max_iter=1000,
        random_state=42
    )
    model.fit(X, y)
    
    # Get absolute coefficients
    coefficients = pd.Series(
        np.abs(model.coef_[0]),
        index=X.columns
    ).sort_values(ascending=False)
    
    # Select top k features with non-zero coefficients
    non_zero_coefs = coefficients[coefficients > 0]
    selected_features = non_zero_coefs.head(k).index.tolist()
    
    return selected_features, coefficients

# Apply Lasso
lasso_features, lasso_coefs = lasso_embedded(X_train_scaled, y_train, k=k_features, C=0.1)

print(f"Top {k_features} features by Lasso:")
for i, feat in enumerate(lasso_features, 1):
    print(f"  {i}. {feat}: {lasso_coefs[feat]:.4f}")
print(f"\nNumber of zero coefficients: {np.sum(lasso_coefs == 0)}")

### 4.2 Tree-Based Feature Importance

Random forests and gradient boosting trees calculate feature importance based on how much each feature decreases impurity.

In [ ]:
# Purpose: Use tree-based feature importance (Gini importance)
# Key Concept: Importance = total reduction in impurity from splits on that feature

from sklearn.ensemble import RandomForestClassifier

def tree_importance_embedded(X, y, k=10):
    """
    Select features using Random Forest importance.
    
    Parameters
    ----------
    X : DataFrame
        Feature matrix
    y : array
        Target variable
    k : int
        Number of features to select
    
    Returns
    -------
    selected_features : list
        Names of selected features
    importances : Series
        Feature importance scores
    """
    # Train Random Forest
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X, y)
    
    # Get feature importances
    importances = pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)
    
    # Select top k features
    selected_features = importances.head(k).index.tolist()
    
    return selected_features, importances

# Apply tree importance
tree_features, tree_importances = tree_importance_embedded(X_train_scaled, y_train, k=k_features)

print(f"Top {k_features} features by Random Forest:")
for i, feat in enumerate(tree_features, 1):
    print(f"  {i}. {feat}: {tree_importances[feat]:.4f}")

## 5. Method Comparison

Now let's compare all methods on the same task using consistent evaluation.

### 5.1 Evaluate All Feature Subsets

Train a model on each feature subset and compare performance.

In [ ]:
# Purpose: Compare all methods using consistent evaluation
# Key Concept: Use same model and CV strategy for fair comparison

def evaluate_feature_subset(X_train, X_test, y_train, y_test, features, method_name):
    """
    Evaluate a feature subset using logistic regression.
    
    Parameters
    ----------
    X_train, X_test : DataFrame
        Train and test feature matrices
    y_train, y_test : array
        Train and test target variables
    features : list
        Selected feature names
    method_name : str
        Name of feature selection method
    
    Returns
    -------
    results : dict
        Dictionary with performance metrics
    """
    # Select features
    X_train_subset = X_train[features]
    X_test_subset = X_test[features]
    
    # Train model
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_subset, y_train)
    
    # Evaluate
    train_score = model.score(X_train_subset, y_train)
    test_score = model.score(X_test_subset, y_test)
    cv_score = cross_val_score(
        model, X_train_subset, y_train, cv=5, scoring='accuracy'
    ).mean()
    
    return {
        'Method': method_name,
        'N_Features': len(features),
        'Train_Acc': train_score,
        'CV_Acc': cv_score,
        'Test_Acc': test_score
    }

# Evaluate all methods
results = []

# Baseline: all features
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    X_train_scaled.columns.tolist(), 'Baseline (All)'
))

# Filter methods
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    corr_features, 'Correlation'
))
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    mi_features, 'Mutual Info'
))

# Wrapper methods
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    rfe_features, 'RFE'
))
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    forward_features, 'Forward Selection'
))

# Embedded methods
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    lasso_features, 'Lasso'
))
results.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    tree_features, 'Random Forest'
))

# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.round(4)

print("Feature Selection Method Comparison")
print("=" * 70)
print(results_df.to_string(index=False))
print("\nBest method (by test accuracy):")
best_idx = results_df['Test_Acc'].idxmax()
print(f"  {results_df.loc[best_idx, 'Method']}: {results_df.loc[best_idx, 'Test_Acc']:.4f}")

### 5.2 Visualize Comparison

Create visualizations to understand the trade-offs between methods.

In [ ]:
# Purpose: Visualize performance comparison across methods
# Key Concept: Look for methods with high test accuracy and low overfitting

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Accuracy comparison
methods = results_df['Method']
x = np.arange(len(methods))
width = 0.25

axes[0].bar(x - width, results_df['Train_Acc'], width, label='Train', alpha=0.8)
axes[0].bar(x, results_df['CV_Acc'], width, label='CV', alpha=0.8)
axes[0].bar(x + width, results_df['Test_Acc'], width, label='Test', alpha=0.8)
axes[0].set_xlabel('Method', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontsize=14)
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.85, 1.0])

# Plot 2: Overfitting analysis (train - test gap)
overfit_gap = results_df['Train_Acc'] - results_df['Test_Acc']
colors = ['red' if gap > 0.05 else 'green' for gap in overfit_gap]
axes[1].barh(methods, overfit_gap, color=colors, alpha=0.7)
axes[1].axvline(x=0.05, color='orange', linestyle='--', label='5% threshold')
axes[1].set_xlabel('Train - Test Accuracy Gap', fontsize=12)
axes[1].set_title('Overfitting Analysis', fontsize=14)
axes[1].legend()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - Lower train-test gap indicates better generalization")
print("  - Red bars show potential overfitting (gap > 5%)")
print("  - Green bars show good generalization (gap <= 5%)")

### 5.3 Feature Overlap Analysis

Which features are consistently selected across methods?

In [ ]:
# Purpose: Analyze feature selection consensus across methods
# Key Concept: Features selected by multiple methods are likely important

# Create feature selection matrix
all_methods = {
    'Correlation': corr_features,
    'Mutual_Info': mi_features,
    'RFE': rfe_features,
    'Forward': forward_features,
    'Lasso': lasso_features,
    'RF': tree_features
}

# Count how many methods selected each feature
feature_counts = {}
for feature in X.columns:
    count = sum(1 for features in all_methods.values() if feature in features)
    feature_counts[feature] = count

# Convert to Series and sort
feature_counts = pd.Series(feature_counts).sort_values(ascending=False)

# Find consensus features (selected by >= 4 methods)
consensus_features = feature_counts[feature_counts >= 4].index.tolist()

print(f"Feature Selection Consensus Analysis")
print("=" * 70)
print(f"\nConsensus features (selected by >= 4 methods): {len(consensus_features)}")
for feat in consensus_features:
    print(f"  - {feat}: selected by {feature_counts[feat]}/6 methods")

# Visualize
plt.figure(figsize=(12, 6))
top_features = feature_counts.head(15)
plt.barh(range(len(top_features)), top_features.values, color='steelblue', edgecolor='black')
plt.yticks(range(len(top_features)), top_features.index)
plt.xlabel('Number of Methods Selecting Feature', fontsize=12)
plt.title('Feature Selection Consensus (Top 15)', fontsize=14)
plt.axvline(x=4, color='red', linestyle='--', label='Consensus threshold (4/6)')
plt.legend()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. When to Use Genetic Algorithms

### Key Insight: GAs excel when:

1. **Large search space**: Too many feature combinations for exhaustive search
2. **Feature interactions**: Complex non-linear relationships between features
3. **Multi-objective**: Need to balance accuracy, interpretability, cost
4. **Flexible fitness**: Custom constraints or domain knowledge
5. **Wrapper limitations**: RFE too slow, forward/backward selection get stuck in local optima

### GA Advantages:
- Explores multiple solutions simultaneously (population-based)
- Escapes local optima through mutation and crossover
- Customizable fitness function
- Parallelizable

### GA Disadvantages:
- Stochastic (results vary between runs)
- Requires careful parameter tuning
- Computationally expensive for fitness evaluation
- No guarantee of finding global optimum

## 7. Exercises

### Exercise 7.1: Compare with Different k

**Task**: Re-run all feature selection methods with k=5 features instead of k=10. Which method benefits most from using fewer features?

**Expected Output**: A comparison table similar to Section 5.1 but with 5 features.

**Hints**:
<details>
<summary>Hint 1</summary>
You can reuse all the functions defined earlier by just changing the k parameter.
</details>

<details>
<summary>Hint 2</summary>
Compare the test accuracy improvement (if any) when reducing from 10 to 5 features.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
k_new = 5

# TODO: Rerun all feature selection methods with k=5
# TODO: Create comparison table
# TODO: Calculate improvement from k=10 to k=5

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_71():
    assert 'k_new' in dir(), "Define k_new = 5"
    assert k_new == 5, "k_new should be 5"
    print("✓ Exercise 7.1 setup correct!")
    print("Note: Complete the analysis and comparison manually")

test_exercise_71()

### Exercise 7.2: Stability Analysis

**Task**: Evaluate the stability of each method by running it 5 times with different random seeds and measuring how often the same features are selected.

**Expected Output**: A stability score (0-1) for each method, where 1 means always selects the same features.

**Hints**:
<details>
<summary>Hint 1</summary>
Use different random_state values (42, 43, 44, 45, 46) for train_test_split.
</details>

<details>
<summary>Hint 2</summary>
Calculate stability as: (number of features selected in all 5 runs) / k
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def measure_stability(X, y, selection_func, k=10, n_runs=5):
    """
    Measure stability of feature selection method.
    
    Returns
    -------
    stability : float
        Proportion of features consistently selected (0-1)
    """
    # TODO: Implement
    pass

# TODO: Test with correlation_filter
# stability_corr = measure_stability(X, y, correlation_filter, k=10)

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_72():
    assert 'measure_stability' in dir(), "Define measure_stability function"
    print("✓ Exercise 7.2 function defined!")
    print("Note: Test your implementation with different methods")

test_exercise_72()

### Exercise 7.3: Custom Fitness Function

**Task**: Design a fitness function for feature selection that balances three objectives:
1. Maximize accuracy
2. Minimize number of features
3. Minimize feature correlation (redundancy)

**Expected Output**: A function that takes a feature subset and returns a fitness score.

**Hints**:
<details>
<summary>Hint 1</summary>
Use weighted sum: fitness = w1*accuracy - w2*(n_features/total_features) - w3*mean_correlation
</details>

<details>
<summary>Hint 2</summary>
Calculate mean pairwise correlation among selected features using X[features].corr()
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def multi_objective_fitness(features, X, y, w1=1.0, w2=0.3, w3=0.2):
    """
    Multi-objective fitness for feature selection.
    
    Parameters
    ----------
    features : list
        Selected feature names
    X : DataFrame
        Full feature matrix
    y : array
        Target variable
    w1, w2, w3 : float
        Weights for accuracy, parsimony, redundancy
    
    Returns
    -------
    fitness : float
        Combined fitness score (higher is better)
    """
    # TODO: Implement
    # 1. Calculate accuracy with cross-validation
    # 2. Calculate parsimony penalty (feature count)
    # 3. Calculate redundancy (mean correlation)
    # 4. Combine with weights
    pass

# Test your function
# test_features = corr_features[:5]
# fitness = multi_objective_fitness(test_features, X_train_scaled, y_train)
# print(f"Fitness: {fitness}")

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_73():
    assert 'multi_objective_fitness' in dir(), "Define multi_objective_fitness function"
    
    # Test with known features
    test_features = X_train_scaled.columns[:3].tolist()
    try:
        fitness = multi_objective_fitness(test_features, X_train_scaled, y_train)
        assert isinstance(fitness, (int, float)), "Fitness should be a number"
        print(f"✓ Exercise 7.3 passed! Test fitness: {fitness:.4f}")
    except:
        print("✗ Function exists but needs implementation")

# Uncomment to test
# test_exercise_73()

### Exercise 7.4: Dimensionality Challenge

**Task**: Create a high-dimensional dataset (1000 features, only 20 informative) and compare the runtime of different feature selection methods.

**Expected Output**: A bar chart showing runtime in seconds for each method.

**Hints**:
<details>
<summary>Hint 1</summary>
Use make_classification with n_features=1000, n_informative=20
</details>

<details>
<summary>Hint 2</summary>
Use time.time() before and after each method to measure runtime
</details>

<details>
<summary>Hint 3</summary>
Skip forward selection for this exercise - it will be too slow!
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
import time
from sklearn.datasets import make_classification

# TODO: Create high-dimensional dataset
# X_high, y_high = make_classification(...)

# TODO: Measure runtime for each method (except forward selection)
# TODO: Create bar chart of runtimes

In [ ]:
# AUTO-GRADED TEST - Do not modify
# ----------------------------------
def test_exercise_74():
    # Check if high-dimensional dataset was created
    if 'X_high' in dir() and 'y_high' in dir():
        assert X_high.shape[1] >= 1000, "Should have at least 1000 features"
        print(f"✓ Exercise 7.4 passed! Dataset shape: {X_high.shape}")
    else:
        print("Create X_high and y_high variables")

# Uncomment to test
# test_exercise_74()

## 8. Solutions

### Solution 7.1: Compare with Different k

In [ ]:
# Solution for Exercise 7.1
k_new = 5

# Rerun all methods with k=5
corr_features_5, _ = correlation_filter(X_train_scaled, y_train, k=k_new)
mi_features_5, _ = mutual_info_filter(X_train_scaled, y_train, k=k_new)
rfe_features_5, _ = rfe_wrapper(X_train_scaled, y_train, k=k_new)
lasso_features_5, _ = lasso_embedded(X_train_scaled, y_train, k=k_new)
tree_features_5, _ = tree_importance_embedded(X_train_scaled, y_train, k=k_new)

# Evaluate
results_5 = []
results_5.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    corr_features_5, 'Correlation (k=5)'
))
results_5.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    mi_features_5, 'Mutual Info (k=5)'
))
results_5.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    rfe_features_5, 'RFE (k=5)'
))
results_5.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    lasso_features_5, 'Lasso (k=5)'
))
results_5.append(evaluate_feature_subset(
    X_train_scaled, X_test_scaled, y_train, y_test,
    tree_features_5, 'Random Forest (k=5)'
))

results_df_5 = pd.DataFrame(results_5).round(4)
print("Results with k=5 features:")
print(results_df_5.to_string(index=False))

### Solution 7.2: Stability Analysis

In [ ]:
# Solution for Exercise 7.2
def measure_stability(X, y, selection_func, k=10, n_runs=5):
    """
    Measure stability of feature selection method.
    """
    from collections import Counter
    
    all_selected = []
    
    for seed in range(42, 42 + n_runs):
        # Create different train/test splits
        X_tr, _, y_tr, _ = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y
        )
        
        # Apply feature selection
        features, _ = selection_func(X_tr, y_tr, k=k)
        all_selected.extend(features)
    
    # Count occurrences
    feature_counts = Counter(all_selected)
    
    # Calculate stability: fraction of features selected in all runs
    stable_features = sum(1 for count in feature_counts.values() if count == n_runs)
    stability = stable_features / k
    
    return stability

# Test stability of correlation filter
stability_corr = measure_stability(X, y, correlation_filter, k=10, n_runs=5)
print(f"Correlation filter stability: {stability_corr:.2f}")

### Solution 7.3: Custom Fitness Function

In [ ]:
# Solution for Exercise 7.3
def multi_objective_fitness(features, X, y, w1=1.0, w2=0.3, w3=0.2):
    """
    Multi-objective fitness for feature selection.
    """
    if len(features) == 0:
        return -999  # Invalid
    
    # Objective 1: Accuracy
    X_subset = X[features]
    model = LogisticRegression(max_iter=1000, random_state=42)
    accuracy = cross_val_score(model, X_subset, y, cv=3, scoring='accuracy').mean()
    
    # Objective 2: Parsimony (fewer features is better)
    parsimony_penalty = len(features) / X.shape[1]
    
    # Objective 3: Redundancy (lower correlation is better)
    if len(features) > 1:
        corr_matrix = X[features].corr().abs()
        # Get upper triangle (exclude diagonal)
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        mean_correlation = upper_triangle.stack().mean()
    else:
        mean_correlation = 0
    
    # Combined fitness (higher is better)
    fitness = w1 * accuracy - w2 * parsimony_penalty - w3 * mean_correlation
    
    return fitness

# Test
test_features = corr_features[:5]
fitness = multi_objective_fitness(test_features, X_train_scaled, y_train)
print(f"Multi-objective fitness: {fitness:.4f}")

## 9. Summary

### Key Takeaways

1. **Three Categories**: Filter (fast, model-agnostic), Wrapper (accurate, slow), Embedded (balanced)
2. **No Universal Best**: Method choice depends on dataset size, feature count, and goals
3. **Filter Methods**: Best for initial screening and high-dimensional data
4. **Wrapper Methods**: Best when model performance is critical and compute time is available
5. **Embedded Methods**: Best for balance between performance and efficiency
6. **Genetic Algorithms**: Excel at exploring large search spaces with complex constraints

### When to Use Each Method

| Method | Use When | Avoid When |
|--------|----------|------------|
| Correlation | Linear relationships, quick screening | Non-linear patterns |
| Mutual Info | Non-linear relationships, exploratory | Need interpretability |
| RFE | Model performance critical | Very high dimensions |
| Forward/Backward | Small feature sets | Large feature sets |
| Lasso | Linear models, sparse solutions | Non-linear patterns |
| Tree Importance | Non-linear patterns, interactions | Linear patterns |
| **Genetic Algorithms** | Complex search spaces, custom constraints | Simple problems |

### What's Next

In **Module 1**, we'll build genetic algorithms from scratch:
- Binary encoding for feature selection
- Selection operators (tournament, roulette wheel)
- Crossover operators (single-point, two-point, uniform)
- Mutation strategies
- Fitness function design

### Additional Resources

- **scikit-learn Feature Selection**: https://scikit-learn.org/stable/modules/feature_selection.html
- **"An Introduction to Variable and Feature Selection"** by Guyon & Elisseeff (2003)
- **MLxtend library**: http://rasbt.github.io/mlxtend/user_guide/feature_selection/
- **Feature Engineering and Selection** (book): http://www.feat.engineering/